# Lab Assignment 4 — Logistic Regression
## Breast Cancer Wisconsin (Diagnostic) Dataset

**Course:** CSET343 — AI in Healthcare  
**Objective:** Preprocess a real-world medical dataset, train a Logistic Regression model for binary classification, and evaluate it using standard classification metrics.

**Dataset:** Breast Cancer Wisconsin (Diagnostic) — UCI Machine Learning Repository

> **Note:** The assignment refers to the target as `Outcome`, but the UCI Diagnostic dataset uses `Diagnosis`, where `M = Malignant` and `B = Benign`. The notebook therefore uses `Diagnosis` as the target variable.


## Dataset Information

The UCI Breast Cancer Wisconsin (Diagnostic) dataset contains **569 instances and 30 real-valued features**. The features are computed from digitized images of fine needle aspirates of breast masses. The target variable is `Diagnosis`.

Source: UCI Machine Learning Repository  
https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic


## Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


## Task 1 — Data Acquisition and Exploration

### 1.1 Download and Load the Dataset

In [ ]:
# Download the dataset from the UCI Machine Learning Repository
breast_cancer = fetch_ucirepo(id=17)

# Separate features and target
X = breast_cancer.data.features.copy()
y = breast_cancer.data.targets.copy()

# Combine them for exploration
df = pd.concat([X, y], axis=1)

print("Dataset shape:", df.shape)
display(df.head())


### 1.2 Basic Dataset Information

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)


### 1.3 Summary Statistics

In [ ]:
# Summary statistics for all numerical features
display(X.describe().T)

# Additional median values
median_values = X.median().to_frame(name="median")
display(median_values)


### 1.4 Check Missing Values

In [ ]:
missing_values = df.isnull().sum()

print("Missing values in each column:")
display(missing_values[missing_values > 0])

if missing_values.sum() == 0:
    print("No missing values found in the dataset.")


### 1.5 Check Duplicate Records

In [ ]:
duplicate_count = df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)

if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicates removed.")
else:
    print("No duplicate rows found.")


### 1.6 Detect Outliers Using the IQR Method

In [ ]:
# Detect potential outliers using the Interquartile Range (IQR)
Q1 = X.quantile(0.25)
Q3 = X.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_counts = ((X < lower_bound) | (X > upper_bound)).sum().sort_values(ascending=False)

print("Potential outliers by feature:")
display(outlier_counts.to_frame(name="outlier_count"))


### Outlier Handling

Potential outliers are not automatically deleted because unusually large measurements can represent genuine medical observations. For the model pipeline, the feature values will be clipped to IQR-based limits calculated from the **training data only**. This avoids deleting observations and prevents test-set information from influencing preprocessing.


### 1.7 Visualize Feature Distributions

In [ ]:
# Histograms for selected important features
selected_features = ["radius1", "texture1", "perimeter1", "area1", "concavity1", "concave_points1"]

X[selected_features].hist(figsize=(14, 9), bins=25, edgecolor="black")
plt.suptitle("Distribution of Selected Features", fontsize=16)
plt.tight_layout()
plt.show()


### 1.8 Box Plots for Outlier Visualization

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=X[selected_features])
plt.xticks(rotation=45)
plt.title("Box Plots of Selected Features")
plt.tight_layout()
plt.show()


### 1.9 Correlation Heatmap

In [ ]:
plt.figure(figsize=(16, 12))
correlation_matrix = X.corr()

sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    center=0,
    linewidths=0.2
)

plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()


### 1.10 Class Distribution

In [ ]:
class_counts = y.iloc[:, 0].value_counts()

print("Class distribution:")
display(class_counts)

plt.figure(figsize=(6, 4))
sns.countplot(x=y.iloc[:, 0])
plt.title("Diagnosis Class Distribution")
plt.xlabel("Diagnosis")
plt.ylabel("Number of Samples")
plt.show()


### Task 1 — Class Distribution Discussion

The dataset contains two diagnosis classes: **B (Benign)** and **M (Malignant)**. The class counts above should be checked before model training.

If one class has substantially more samples than the other, accuracy alone may give an incomplete picture. Precision, recall, F1-score, confusion matrix and ROC-AUC are therefore also used in Task 3. Stratified train-test splitting is used in Task 2 to preserve approximately the same class proportion in both sets.


## Task 2 — Data Preprocessing

### 2.1 Encode the Target Variable as Binary

In [ ]:
# Encode B = 0 (Benign), M = 1 (Malignant)
y_binary = y.iloc[:, 0].map({"B": 0, "M": 1})

print("Encoded target values:")
print(y_binary.value_counts().sort_index())

print("\nMapping:")
print("B (Benign)    -> 0")
print("M (Malignant) -> 1")


### 2.2 Remove the ID Column

In [ ]:
# ID is an identifier, not a clinical measurement, so it is excluded from modeling.
X_model = X.drop(columns=["ID"], errors="ignore")

print("Features used for modeling:", X_model.shape[1])
print(X_model.columns.tolist())


### 2.3 Train-Test Split Using Stratified Sampling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_binary,
    test_size=0.20,
    random_state=42,
    stratify=y_binary
)

print("Training set:", X_train.shape)
print("Testing set :", X_test.shape)

print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True))

print("\nTesting class proportions:")
print(y_test.value_counts(normalize=True))


### 2.4 Handle Outliers Using Training-Set IQR Limits

In [ ]:
# Calculate IQR limits using ONLY the training data
Q1_train = X_train.quantile(0.25)
Q3_train = X_train.quantile(0.75)
IQR_train = Q3_train - Q1_train

lower_train = Q1_train - 1.5 * IQR_train
upper_train = Q3_train + 1.5 * IQR_train

# Clip both training and testing data using the training-derived limits
X_train_clean = X_train.clip(lower=lower_train, upper=upper_train, axis=1)
X_test_clean = X_test.clip(lower=lower_train, upper=upper_train, axis=1)

print("Outlier handling completed using training-set IQR limits.")


### 2.5 Standardize the Features

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_clean)
X_test_scaled = scaler.transform(X_test_clean)

print("Standardization completed.")
print("Training data mean (approximately):", X_train_scaled.mean())
print("Training data standard deviation (approximately):", X_train_scaled.std())


## Task 3 — Logistic Regression and Basic Evaluation

### 3.1 Train the Logistic Regression Model

In [ ]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully.")


### 3.2 Generate Predictions

In [ ]:
y_pred = logistic_model.predict(X_test_scaled)
y_prob = logistic_model.predict_proba(X_test_scaled)[:, 1]

print("First 10 predicted classes:", y_pred[:10])
print("First 10 malignant probabilities:", y_prob[:10])


### 3.3 Accuracy, Precision, Recall and F1-Score

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")


### 3.4 Precision, Recall and F1-Score for Both Classes

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["Benign (0)", "Malignant (1)"],
    digits=4
))


### 3.5 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Benign", "Malignant"]
)

disp.plot(cmap="Blues")
plt.title("Confusion Matrix — Logistic Regression")
plt.show()

print("Confusion Matrix:")
print(cm)


### 3.6 ROC Curve and AUC Score

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc_score:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

print(f"ROC-AUC Score: {auc_score:.4f}")


### 3.7 Final Evaluation Summary

In [ ]:
results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"],
    "Score": [accuracy, precision, recall, f1, auc_score]
})

display(results)


### 3.8 Interpretation of Results

Use the values produced above to write the final interpretation.

- **Accuracy:** Shows the overall proportion of correctly classified cases.
- **Precision:** Shows how many cases predicted as malignant were actually malignant.
- **Recall (Sensitivity):** Shows how many actual malignant cases were correctly identified.
- **F1-score:** Provides a combined measure of precision and recall.
- **Confusion matrix:** Shows the numbers of true positives, true negatives, false positives and false negatives.
- **ROC-AUC:** Measures the model's ability to distinguish between benign and malignant cases across classification thresholds.

For a medical diagnosis problem, **false negatives are particularly important to examine**, because a malignant case incorrectly predicted as benign represents a missed positive diagnosis. Therefore, recall for the malignant class should be considered alongside the other metrics rather than relying only on accuracy.


## Conclusion

The Breast Cancer Wisconsin (Diagnostic) dataset was loaded from the UCI Machine Learning Repository. The data was explored using summary statistics and visualizations, missing values and duplicates were checked, potential outliers were identified and handled using training-set IQR limits, and the features were standardized.

A Logistic Regression classifier was then trained using a stratified 80/20 train-test split. The model was evaluated using accuracy, precision, recall, F1-score, confusion matrix and ROC-AUC, providing a more complete view of its classification performance for the two diagnosis classes.
